[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C03_LLM_Evals_Course/04_llm_judge/04_llm_judge.ipynb)

# 04 · LLM-as-a-Judge 深剖 —— 把 judge 当测量仪器来实测

配套讲解：`04_讲解.html`。本 notebook 用一个**小模型 judge**（`Qwen/Qwen2.5-1.5B-Instruct`）在 8 对内嵌回答上复现 judge 的典型病理，并亲手实现去偏与 meta-evaluation 工具。

**四个实验**

1. **Position bias**：每对回答按正反两个顺序各判一次，统计 swap 后判决翻转率与"位置 1 胜率"。
2. **Length bias**：在"gold = 短回答更好"的子集上，看 judge 是否系统性偏长。
3. **与人类一致性**：judge 判决 vs 人工 gold 标签 → 一致率与 Cohen's $\kappa$。
4. **Rubric 对照**：bare prompt vs 三维 rubric prompt，对比 $\kappa$ 变化。

**三个练习**：① 从头实现 `cohens_kappa`；② position-debiased 判决；③ 多 judge 多数票聚合（含 tie 处理）。

**运行模式**（下个 cell 自动检测）

| 模式 | 条件 | 资源 |
|---|---|---|
| `local` | 装有 `transformers` + `torch` | 首次约 3GB 模型下载；CPU 可跑（每次判决数秒～数十秒，全程约 50 次判决），GPU/MPS 显著更快 |
| `api` | 手动设 `USE_API_IF_AVAILABLE=True` 且已配置 `OPENAI_API_KEY` | 用 `gpt-4o-mini` 当 judge，约 50 次调用 |
| `mock` | 以上都不可用时自动回退 | 零资源、完全确定性的"有偏 judge 模拟器"（原理见后文，附诚实声明） |


In [ ]:
import os, math, random, hashlib

# ---- 运行模式选择 --------------------------------------------------------
# local : 本地小模型 judge（Qwen/Qwen2.5-1.5B-Instruct）
# api   : OPENAI_API_KEY 存在且手动开启时，用 gpt-4o-mini 当 judge
# mock  : 无模型环境的确定性回退（有偏 judge 模拟器）
USE_API_IF_AVAILABLE = False   # 改成 True 且 export OPENAI_API_KEY 才会走 API

MODE = "mock"
tokenizer = model = None

if USE_API_IF_AVAILABLE and os.environ.get("OPENAI_API_KEY"):
    MODE = "api"
else:
    try:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, torch_dtype="auto", device_map="auto")
        model.eval()
        MODE = "local"
    except Exception as e:
        print(f"[回退] 本地模型不可用（{type(e).__name__}: {e}），改用 mock judge 模拟器。")
        MODE = "mock"

print(f"judge 运行模式: {MODE}")

In [ ]:
# 8 对回答：同一问题的 A/B 两份回答 + 人工 gold 偏好标签
#   short_better       : 一长一短，但短的才是好回答（长的啰嗦/跑题/有错）
#   objective          : 有客观对错
#   style_vs_substance : 排版花哨但事实错误 vs 朴素但正确
PAIRS = [
    dict(tag="short_better", gold="A",
         q="法国的首都是哪座城市？",
         ans_a="巴黎。",
         ans_b="这是一个非常好的问题！法国是欧洲历史最悠久、文化最丰富的国家之一，"
               "拥有灿烂的艺术、美食与建筑遗产。它的首都是里昂，那里坐落着举世闻名的"
               "卢浮宫和埃菲尔铁塔，每年吸引数千万游客慕名而来。"),
    dict(tag="short_better", gold="B",
         q="Python 里如何反转列表 lst？",
         ans_a="反转列表的方法非常多，这恰恰体现了 Python 设计哲学的灵活性。你可以写"
               "一个 for 循环，把元素逐个 append 到新列表开头，复杂度 O(n^2)；也可以"
               "先把列表转换成字符串再逐字符处理；还可以借助第三方库。总之条条大路"
               "通罗马，选择哪种完全取决于个人编码风格。",
         ans_b="lst[::-1] 返回反转后的新列表；lst.reverse() 原地反转。"),
    dict(tag="objective", gold="A",
         q="17 × 24 等于多少？",
         ans_a="17 × 24 = 408。",
         ans_b="17 × 24 = 398。"),
    dict(tag="objective", gold="B",
         q="光在真空中的传播速度大约是多少？",
         ans_a="大约 3 × 10^7 m/s。",
         ans_b="大约 3 × 10^8 m/s（精确值 299,792,458 m/s）。"),
    dict(tag="style_vs_substance", gold="B",
         q="二分查找的时间复杂度是多少？为什么？",
         ans_a="## 二分查找复杂度分析\n\n二分查找是计算机科学中最优雅的算法之一。\n\n"
               "**结论：时间复杂度为 O(n log n)。**\n\n原因：每一轮都需要先对当前区间"
               "排序（O(n)），再取中点比较（O(log n)），两者相乘即得 O(n log n)。"
               "这也是它比线性查找更快的根本原因。",
         ans_b="O(log n)。每次比较把搜索区间减半，长度 n 的区间约经 log2(n) 次减半后"
               "只剩一个元素。前提：数组已排序。"),
    dict(tag="style_vs_substance", gold="A",
         q="为什么白天的天空是蓝色的？",
         ans_a="因为瑞利散射：大气分子尺度远小于可见光波长，散射强度近似与波长四次方"
               "成反比，短波长的蓝光被散射得最强，从四面八方进入人眼。",
         ans_b="🌌 多么浪漫的问题！**答案：天空是大海的镜子。**\n\n大气层就像一面巨大"
               "的镜子，把海洋的蔚蓝反射到了我们头顶。这就是为什么沿海地区的天空总是"
               "格外湛蓝，而内陆沙漠的天空常常发白——因为那里没有海可以照映！"),
    dict(tag="short_better", gold="A",
         q="成年人的正常体温大约是多少？",
         ans_a="口腔测量约 36.3–37.2°C，腋下略低。",
         ans_b="体温是反映健康状况的重要生理指标，受运动、饮食、情绪、昼夜节律、测量"
               "部位等多种因素影响，每个人都不尽相同。建议您购买一支经过认证的体温计，"
               "在安静状态下定期测量并做好记录，如有任何疑虑请及时咨询专业医生，"
               "切勿自行下结论。"),
    dict(tag="short_better", gold="B",
         q="HTTP 状态码 404 代表什么含义？",
         ans_a="404 是最常见的状态码之一，它表示服务器内部出现了错误，一般是后端代码"
               "抛出了未捕获的异常所致。遇到 404 时建议先清空浏览器缓存，再重启路由器，"
               "如果仍未解决，就需要联系网站管理员修复服务器上的 bug。",
         ans_b="404 Not Found：服务器找不到客户端请求的资源（URL 对应的内容不存在）。"),
]

def gold_ans(p):  return p["ans_a"] if p["gold"] == "A" else p["ans_b"]
def other_ans(p): return p["ans_b"] if p["gold"] == "A" else p["ans_a"]

print(f"{len(PAIRS)} 对回答；gold 分布:",
      {g: [p["gold"] for p in PAIRS].count(g) for g in "AB"})
for i, p in enumerate(PAIRS, 1):
    print(f"P{i} [{p['tag']:<18}] gold={p['gold']}  "
          f"len(gold)={len(gold_ans(p)):>3}  len(另一份)={len(other_ans(p)):>3}")

## Judge 的设计：prompt、解析与 mock 模拟器

两种 pairwise judge prompt：

- **bare**：直接问"哪份更好，只输出 1 或 2"——把质量的定义权完全交给 judge 的内部偏好。
- **rubric**：三维准则（**事实正确性 → 针对性 → 简洁性**），并采用**链式评分**——先逐条列出每份回答的事实错误，最后单独一行输出编号。rubric 里显式写明"更长不等于更好"。

判决解析沿用模块 03 的教训：对自由文本做稳健解析（从最后一行向前找唯一出现的 1/2），解析失败返回 `None` 而不是瞎猜。

> **关于 mock 模式的诚实声明**：mock 不是语言模型。它查表知道每对的 gold（仅教学用途），再叠加四个可控分量：**质量信号 + 长度偏差 + 位置偏差 + 确定性噪声**。bare 配置故意调成"重长度、重位置"，rubric 配置"重质量"——以便在零资源环境下复现真实 judge 的典型病理与 rubric 的效果。**mock 模式产出的数字只用于理解实验逻辑，不构成对任何真实模型的结论**；有条件请切换 `local` / `api` 模式重跑全部实验。

### 实验 1 & 2 说明

- **实验 1（position bias）**：每对回答按 `(A,B)` 与 `(B,A)` 两个顺序各判一次。两次判决若指向**不同的回答**，记一次 **swap 翻转**——这说明判决跟着"位置"走而不是跟着"内容"走。同时统计全部判决中"位置 1 获胜"的比例，无偏 judge 应 ≈ 50%。
- **实验 2（length bias）**：取"gold 明显更短"的子集（另一份长度 > 1.5 × gold 长度）。理想 judge 在该子集上的**选长率应为 0%**；实测显著高于 0 即为长度偏差的直接证据（参照 [Dubois 2024] 的动机）。

In [ ]:
BARE_TEMPLATE = '''你是一位公正的评审，需要比较两份针对同一问题的回答。

[问题]
{q}

[回答1]
{a1}

[回答2]
{a2}

哪一份回答更好？不要解释，只输出一个字符：1 或 2。'''

RUBRIC_TEMPLATE = '''你是一位公正的评审，需要比较两份针对同一问题的回答。

[问题]
{q}

[回答1]
{a1}

[回答2]
{a2}

请按以下三个维度评判（按重要性排序）：
1. 事实正确性：先逐条列出每份回答中的事实错误（若无则写"无"）。
2. 针对性：是否直接、完整地回答了所问的问题？
3. 简洁性：相同信息量下更简洁者优。注意：更长不等于更好。

先完成上述分析，最后单独一行只输出获胜回答的编号：1 或 2。'''


def parse_verdict(text):
    '''从 judge 的自由文本输出解析判决 -> "1" / "2" / None（解析失败不瞎猜）。'''
    if not text:
        return None
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    for line in reversed(lines):
        digits = {c for c in line if c in "12"}
        if len(digits) == 1:
            return digits.pop()
    return None


def local_generate(prompt, max_new_tokens):
    msgs = [{"role": "user", "content": prompt}]
    # transformers 5.x：apply_chat_template(return_tensors="pt") 返回 BatchEncoding（无 .shape），
    # 故用 return_dict=True 取 dict，generate(**inputs) 传入，并以 inputs["input_ids"] 计算 prompt 长度。
    inputs = tokenizer.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    prompt_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True)


def api_generate(prompt, max_new_tokens):
    from openai import OpenAI
    client = OpenAI()  # 读取环境变量 OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_new_tokens, temperature=0)
    return resp.choices[0].message.content


# ---------------- mock judge 模拟器（仅教学；见上方诚实声明） ----------------
GOLD_TEXT = {p["q"]: gold_ans(p) for p in PAIRS}

def _unit(*parts):
    '''确定性伪随机数 [0,1)：md5(输入) -> 均匀值，保证结果可复现。'''
    h = hashlib.md5("||".join(parts).encode("utf-8")).hexdigest()
    return int(h[:8], 16) / 16**8

def mock_verdict(q, a1, a2, rubric):
    if rubric:   # 重质量、轻长度/位置 —— 模拟 rubric 的约束效果
        w_quality, w_len, w_pos, w_noise = 0.75, 0.25, 0.15, 0.70
    else:        # 重长度、重位置 —— 模拟 bare judge 的典型病理
        w_quality, w_len, w_pos, w_noise = 0.50, 0.85, 0.15, 0.50
    scores = []
    for pos, ans in enumerate((a1, a2)):
        s  = w_quality * (1.0 if ans == GOLD_TEXT.get(q) else 0.0)
        s += w_len * min(len(ans), 120) / 120
        s += w_pos if pos == 0 else 0.0
        s += w_noise * _unit(q, ans[:24], str(rubric), str(pos))
        scores.append(s)
    return "1" if scores[0] >= scores[1] else "2"


def judge_once(q, ans_1, ans_2, rubric=False):
    '''统一入口：返回 "1"（第一份好）/ "2"（第二份好）/ None（解析失败）。'''
    if MODE == "mock":
        return mock_verdict(q, ans_1, ans_2, rubric)
    template = RUBRIC_TEMPLATE if rubric else BARE_TEMPLATE
    prompt = template.format(q=q, a1=ans_1, a2=ans_2)
    text = (local_generate(prompt, 256 if rubric else 8) if MODE == "local"
            else api_generate(prompt, 300 if rubric else 8))
    return parse_verdict(text)


# 冒烟测试：P1（巴黎 vs 啰嗦且答错的"里昂"）
print("P1 判决（A 在位置1）:", judge_once(PAIRS[0]["q"], PAIRS[0]["ans_a"], PAIRS[0]["ans_b"]))

In [ ]:
# ============ 实验 1：position bias —— swap 后判决翻转率 ============
records = []
for p in PAIRS:
    v_ab = judge_once(p["q"], p["ans_a"], p["ans_b"])   # 顺序1: A 在位置1
    v_ba = judge_once(p["q"], p["ans_b"], p["ans_a"])   # 顺序2: B 在位置1
    records.append(dict(p=p, v_ab=v_ab, v_ba=v_ba,
                        pick_ab={"1": "A", "2": "B"}.get(v_ab),   # 顺序1 指向的回答
                        pick_ba={"1": "B", "2": "A"}.get(v_ba)))  # 顺序2 指向的回答

valid = [r for r in records if r["pick_ab"] and r["pick_ba"]]
flips = [r for r in valid if r["pick_ab"] != r["pick_ba"]]
all_v = [v for r in records for v in (r["v_ab"], r["v_ba"]) if v]
pos1  = sum(v == "1" for v in all_v)

if valid:
    print(f"swap 后判决翻转率: {len(flips)}/{len(valid)} = {len(flips)/len(valid):.0%}")
    print(f"位置 1 胜率      : {pos1}/{len(all_v)} = {pos1/len(all_v):.0%}   （无偏应≈50%）")
print("\n逐对判决（顺序1 = A 在位置1）:")
for r in records:
    flip = r["pick_ab"] and r["pick_ba"] and r["pick_ab"] != r["pick_ba"]
    print(f'  {r["p"]["q"][:16]:<18} 顺序1→{r["pick_ab"]}  顺序2→{r["pick_ba"]}'
          + ("  ← swap 翻转" if flip else ""))

# ============ 实验 2：length bias —— "gold 更短"子集上的选长率 ============
sub = [r for r in records if 1.5 * len(gold_ans(r["p"])) < len(other_ans(r["p"]))]
picks_longer = picks_total = 0
for r in sub:
    longer = "A" if len(r["p"]["ans_a"]) > len(r["p"]["ans_b"]) else "B"
    for pick in (r["pick_ab"], r["pick_ba"]):
        if pick:
            picks_total += 1
            picks_longer += (pick == longer)

print(f"\n'gold 更短'子集: {len(sub)} 对（gold 全为较短回答 → 理想选长率 0%）")
if picks_total:
    print(f"judge 选了更长回答的比例: {picks_longer}/{picks_total} = {picks_longer/picks_total:.0%}")
    print("显著高于 0% ⇒ 长度偏差。系统级修法见讲解 §3.2 的长度回归校正 [Dubois 2024]。")

## ✏️ 练习 1：实现 `cohens_kappa(y1, y2)`

实验 3 要量化"judge 判决与人工 gold 的对齐度"。原始一致率有个缺陷：偏好分布不均时瞎猜也能得高分。Cohen's $\kappa$ 扣掉"碰巧一致"的期望：

$$\kappa = \frac{p_o - p_e}{1 - p_e},\qquad p_e = \sum_{k} p_{1k}\, p_{2k}$$

其中 $p_o$ 为观测一致率，$p_{1k}, p_{2k}$ 为两位标注者各自给出类别 $k$ 的频率。

**任务**：实现多类别版 `cohens_kappa(y1, y2)`（标签可以是 `"A"/"B"/"tie"` 等任意可哈希值），10 行左右即可。

**提示**：
1. $p_o$ = 逐位置相等的比例；
2. $p_e$ = 对所有出现过的标签求 `(y1 中频率) × (y2 中频率)` 之和；
3. 退化情形 $p_e = 1$（两边都只出现同一个标签）：完全一致返回 `1.0`，否则返回 `0.0`。

自测会与 `sklearn.metrics.cohen_kappa_score` 随机对照（sklearn 只用于对照，不得用于实现；未安装则自动跳过该部分）。

In [ ]:
def cohens_kappa(y1, y2):
    '''两组类别标注的 Cohen's kappa。

    y1, y2 : 等长序列，元素为可哈希标签（如 "A" / "B" / "tie"）
    返回   : float；退化情形 p_e == 1 时，完全一致返回 1.0，否则 0.0
    '''
    assert len(y1) == len(y2) and len(y1) > 0
    # TODO 1: 计算观测一致率 p_o
    # TODO 2: 计算期望一致率 p_e = sum_k p1k * p2k（对 y1、y2 中出现过的所有标签）
    # TODO 3: 处理 p_e == 1 的退化情形，否则返回 (p_o - p_e) / (1 - p_e)
    raise NotImplementedError

In [ ]:
# ---- 练习 1 自测 ----
assert math.isclose(cohens_kappa(["A", "B", "A", "B"], ["A", "B", "A", "B"]), 1.0)
assert math.isclose(cohens_kappa(["A", "A", "B", "B"], ["B", "B", "A", "A"]), -1.0)
assert math.isclose(cohens_kappa(["A", "A", "B", "B"], ["A", "B", "A", "B"]), 0.0)
assert math.isclose(cohens_kappa(["A", "A", "A", "B"], ["A", "A", "B", "B"]), 0.5)
assert math.isclose(cohens_kappa(["A", "A"], ["A", "A"]), 1.0)   # 退化: p_e = 1 且完全一致

try:
    from sklearn.metrics import cohen_kappa_score   # 仅用于对照
    rng = random.Random(0)
    for _ in range(20):
        z1 = [rng.choice(["A", "B", "tie"]) for _ in range(60)]
        z2 = [rng.choice(["A", "B", "tie"]) for _ in range(60)]
        assert math.isclose(cohens_kappa(z1, z2), cohen_kappa_score(z1, z2),
                            abs_tol=1e-8)
    print("sklearn 随机对照 20 组全部一致")
except ImportError:
    print("（未安装 sklearn，跳过对照；手工用例已覆盖）")

print("✅ 练习 1 通过")

## 实验 3 & 4：meta-evaluation 与 rubric 对照

- **实验 3（与人类一致性）**：以顺序 `(A,B)` 的判决作为 judge 答案（**故意不做 swap 消偏**——完成练习 2 后你可以自己对比消偏前后的差异），与 gold 标签计算一致率与 $\kappa$。
- **实验 4（rubric 对照）**：换三维 rubric prompt 把 8 对全部重判，对比 bare / rubric 两套判决的 $\kappa$——rubric 能否把 judge 从"接近碰运气"拉起来？
- ⚠️ 统计提醒（模块 02）：$n=8$ 时 $\kappa$ 的方差很大，本实验只看**方向**；正式 meta-evaluation 需要数百条以上的 gold 标注并报告置信区间。

In [ ]:
# ============ 实验 3：judge vs gold —— 一致率与 Cohen's κ ============
judge_picks = [r["pick_ab"] or "tie" for r in records]   # 解析失败计为 tie
gold_labels = [r["p"]["gold"] for r in records]

agree = sum(j == g for j, g in zip(judge_picks, gold_labels)) / len(gold_labels)
print(f"bare judge 与 gold 一致率 : {agree:.0%}")
try:
    print(f"bare judge Cohen's κ      : {cohens_kappa(judge_picks, gold_labels):.3f}")
except NotImplementedError:
    print("（先完成 ✏️ 练习 1，再重跑本 cell 可看到 κ）")

# ============ 实验 4：rubric 对照 —— bare vs 三维 rubric ============
rubric_picks = []
for p in PAIRS:
    v = judge_once(p["q"], p["ans_a"], p["ans_b"], rubric=True)
    rubric_picks.append({"1": "A", "2": "B"}.get(v) or "tie")

agree_r = sum(j == g for j, g in zip(rubric_picks, gold_labels)) / len(gold_labels)
print(f"\nrubric judge 与 gold 一致率: {agree_r:.0%}")
try:
    k_bare = cohens_kappa(judge_picks, gold_labels)
    k_rub  = cohens_kappa(rubric_picks, gold_labels)
    print(f"κ 对比: bare = {k_bare:.3f}  →  rubric = {k_rub:.3f}")
    print("rubric（维度拆分 + 先核事实再判决）通常显著提升与 gold 的对齐。")
except NotImplementedError:
    print("（先完成 ✏️ 练习 1，再重跑本 cell 可看到 κ 对比）")

## ✏️ 练习 2：position-debiased 判决 `debiased_judge`

实验 1 表明判决会随位置翻转。最便宜的消偏：**两个顺序都问，一致才记胜负，不一致记 tie**（讲解 §3.1 的 swap 流水线）。

**任务**：实现

```python
debiased_judge(judge_fn, question, ans_a, ans_b) -> "A" | "B" | "tie"
```

其中 `judge_fn(question, 第一份回答, 第二份回答)` 返回 `"1"` 或 `"2"`（指**位置**）。

**提示**：
1. 顺序 1 用 `(ans_a, ans_b)` 问一次，顺序 2 用 `(ans_b, ans_a)` 再问一次；
2. 把两个位置判决分别映射回 `"A"/"B"`（注意顺序 2 中位置 1 是 `ans_b`！）；
3. 两次指向同一回答 → 返回该标签；否则返回 `"tie"`。

完成后可以把 `judge_once` 包进来，重算实验 3 的 κ，对比消偏前后的变化。

In [ ]:
def debiased_judge(judge_fn, question, ans_a, ans_b):
    '''position-debiased 判决：两个顺序都问，不一致记 "tie"。

    judge_fn(question, 第一份, 第二份) -> "1" 或 "2"（位置判决）
    返回 "A" / "B" / "tie"
    '''
    # TODO 1: 顺序1 (ans_a, ans_b) 与 顺序2 (ans_b, ans_a) 各问一次
    # TODO 2: 把两个位置判决映射回 "A"/"B"
    # TODO 3: 一致返回该标签，不一致返回 "tie"
    raise NotImplementedError

In [ ]:
# ---- 练习 2 自测（用三个行为已知的 mock judge_fn）----
def _always_first(q, a1, a2):   # 纯位置偏差：永远选位置 1
    return "1"

def _prefer_longer(q, a1, a2):  # 纯长度偏差：永远选更长的
    return "1" if len(a1) >= len(a2) else "2"

def _prefer_408(q, a1, a2):     # 看内容：选含 "408" 的那份
    return "1" if "408" in a1 else "2"

assert debiased_judge(_always_first, "q?", "甲", "乙") == "tie"          # 纯位置偏差 → tie
assert debiased_judge(_prefer_longer, "q?", "很长很长的回答", "短") == "A"
assert debiased_judge(_prefer_longer, "q?", "短", "很长很长的回答") == "B"
assert debiased_judge(_prefer_408, "q?", "答案是 408", "答案是 398") == "A"
assert debiased_judge(_prefer_408, "q?", "答案是 398", "答案是 408") == "B"
print("✅ 练习 2 通过")

## ✏️ 练习 3：多 judge 多数票聚合 `majority_vote`

多 judge ensemble（讲解 §6.3）能平均掉**不相关**的偏差分量并稀释 self-preference [Panickssery 2024]，但聚合规则必须预先定死。

**任务**：实现 `majority_vote(verdicts) -> "A" | "B" | "tie"`，其中 `verdicts` 是各 judge 的判决列表，元素 ∈ `{"A", "B", "tie"}`。

**规则**：
1. 统计 `"A"` 与 `"B"` 的票数，`"tie"` 票不计入任何一方；
2. 票多者胜；票数相同（含全为 tie、空列表）返回 `"tie"`。

In [ ]:
def majority_vote(verdicts):
    '''多 judge 判决聚合（多数票，含 tie 处理）。

    verdicts : 列表，元素 ∈ {"A", "B", "tie"}
    返回     : "A" / "B" / "tie"
    '''
    # TODO: 按上方规则实现（"tie" 票不计入任何一方；平票返回 "tie"）
    raise NotImplementedError

In [ ]:
# ---- 练习 3 自测 ----
assert majority_vote(["A", "A", "B"]) == "A"
assert majority_vote(["B", "tie", "B"]) == "B"
assert majority_vote(["A", "B", "tie"]) == "tie"          # 平票 → tie
assert majority_vote(["tie", "tie", "B"]) == "B"          # tie 票不抵消有效票
assert majority_vote(["A", "A", "B", "B"]) == "tie"
assert majority_vote(["tie", "tie", "tie"]) == "tie"
assert majority_vote(["B"]) == "B"
assert majority_vote([]) == "tie"                          # 边界：空列表
print("✅ 练习 3 通过")

## 📖 参考答案

先自己做，再对照——直接运行参考答案会让前面的自测失去意义。运行某题参考答案后，重跑对应的自测 cell 验证；重跑实验 3/4 的 cell 可补全 κ 输出。

In [ ]:
# ---- 参考答案 1（先自己做，再对照）----
def cohens_kappa(y1, y2):
    assert len(y1) == len(y2) and len(y1) > 0
    y1, y2 = list(y1), list(y2)
    n = len(y1)
    p_o = sum(a == b for a, b in zip(y1, y2)) / n
    p_e = sum((y1.count(k) / n) * (y2.count(k) / n) for k in set(y1) | set(y2))
    if p_e == 1.0:                       # 退化：两边都只有同一个标签
        return 1.0 if p_o == 1.0 else 0.0
    return (p_o - p_e) / (1 - p_e)

In [ ]:
# ---- 参考答案 2（先自己做，再对照）----
def debiased_judge(judge_fn, question, ans_a, ans_b):
    v1 = judge_fn(question, ans_a, ans_b)        # 顺序1: 位置1 = ans_a
    v2 = judge_fn(question, ans_b, ans_a)        # 顺序2: 位置1 = ans_b
    pick1 = "A" if v1 == "1" else "B"
    pick2 = "B" if v2 == "1" else "A"
    return pick1 if pick1 == pick2 else "tie"


In [ ]:
# ---- 参考答案 3（先自己做，再对照）----
def majority_vote(verdicts):
    a, b = verdicts.count("A"), verdicts.count("B")
    if a > b:
        return "A"
    if b > a:
        return "B"
    return "tie"


## 小结

- judge 是**有噪声、有系统偏差的测量仪器**：position bias 与 length bias 你已在实验 1/2 中亲手测出。
- **swap + tie** 是最便宜的位置去偏（练习 2）；长度偏差要靠回归校正（length-controlled AlpacaEval [Dubois 2024]）或 rubric 显式约束。
- judge 必须先被 **meta-evaluate**：逐题对齐用 Cohen's κ（练习 1），系统排名对齐用 Spearman ρ；judge–human 一致率的参照天花板是人–人一致率（MT-Bench：85% vs 81% [Zheng 2023]）。
- **rubric**（维度拆分 + 锚定样例 + 先核事实再判决）能显著提升对齐（实验 4），但 rubric 本身也是 prompt，同样要做敏感性分析（模块 03）。
- **self-preference** [Panickssery 2024]：不要用被测模型的同族当唯一 judge；多 judge 多数票（练习 3）可以稀释——但所有 LLM 共享的偏差（长度、自信语气）ensemble 消不掉。
- 可验证任务（代码执行、精确匹配）**禁用 judge**；高风险结论 judge 只能初筛，必须人审。

**下一站 → [05 · 数据污染与基准饱和](../05_contamination/05_讲解.html)**：当 benchmark 题目混进了训练数据，再严谨的测量也只是测"背诵"——如何检测与防御数据污染。

---
## 🎯 真实数据胶囊题：真实成对数据上的 LLM-judge 位置偏置与去偏

LLM judge 有**位置偏置**：倾向选第一个。用真实红酒成对（质量高者为真赢家）模拟一个带位置偏置的 judge，量化偏置，并用“交换位置再判、取一致”的去偏方法把准确率拉回。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.llm_evals_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(ans): return ans.split("####")[-1].strip().replace(",","")
def shakespeare():
    return open(_f("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt","shake.txt")).read()

import pandas as pd
p=_f("https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv","winequality-red.csv")
df=pd.read_csv(p,sep=";"); q=df["quality"].to_numpy(); rng=np.random.default_rng(0)
pairs=[]
for _ in range(800):
    a,b=rng.integers(0,len(q),2)
    if q[a]!=q[b]: pairs.append((a,b, 0 if q[a]>q[b] else 1))  # 真赢家 index in (a,b)
def biased_judge(qa, qb, first_bonus=0.8):
    # judge 看质量差，但对"第一个"有加成
    s=(qa-qb)+first_bonus
    return 0 if s>0 else 1
print(f"{len(pairs)} 个真实成对比较, judge 对第一个位置有偏置")

**练习**：实现 `debiased_judge(qa, qb, first_bonus)`：分别以 (a在前) 和 (b在前) 各判一次，只有两次一致才采纳、否则判平局(返回 None)。返回赢家在 (a,b) 中的 index 或 None。

In [ ]:
def debiased_judge(qa, qb, first_bonus=0.8):
    # TODO: judge(qa,qb) 和 judge(qb,qa) 都做；一致才返回赢家，否则 None
    raise NotImplementedError


In [ ]:
# 自测：去偏后在"有明确赢家"的对上准确率更高
def naive_acc():
    c=0
    for a,b,w in pairs:
        c += (biased_judge(q[a],q[b])==w)
    return c/len(pairs)
def debiased_acc():
    c=0; n=0
    for a,b,w in pairs:
        r=debiased_judge(q[a],q[b])
        if r is not None: n+=1; c+=(r==w)
    return c/max(n,1)
na, da = naive_acc(), debiased_acc()
assert da > na, f"去偏(交换一致)应提高准确率: {na:.2f}->{da:.2f}"
print(f"位置偏置 judge 准确率 {na:.2f} -> 交换去偏后 {da:.2f} ✓")


### 📖 参考答案

In [ ]:
def debiased_judge(qa, qb, first_bonus=0.8):
    r1 = biased_judge(qa, qb, first_bonus)        # a 在前
    r2 = biased_judge(qb, qa, first_bonus)        # b 在前, 返回0表示"前者(b)"赢
    w1 = 0 if r1==0 else 1                          # 赢家在(a,b)
    w2 = 1 if r2==0 else 0                          # 换算回(a,b)
    return w1 if w1==w2 else None
print("✓ 交换位置取一致，是消 LLM-judge 位置偏置的标准手法")